In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import arviz as az
import bambi as bmb
from sklearn.preprocessing import StandardScaler

from utils import data

In [3]:
df = data.load_cluster_features(min_cluster_size=0)
df = df.to_pandas()

In [6]:
df.columns

Index(['window_id', 'window_idx', 'wn_mid_date', 'wn_prop_sequenced',
       'cluster_id', 'n_sequences', 'n_sequences_minus_one', 'resolution',
       'median_age', 'mean_age', 'age_diversity', 'frac_female',
       'frac_vaccinated', 'simd_decile_mode', 'simd_decile_std',
       'simd_quintile_mode', 'simd_quintile_std', 'overall_zscore',
       'income_zscore', 'employment_zscore', 'education_zscore',
       'health_zscore', 'access_zscore', 'crime_zscore', 'housing_zscore',
       'pango_lineage', 'who_voc', 'wave'],
      dtype='str')

In [12]:
top_lineages = df["pango_lineage"].value_counts().head(10).index
df["lineage_lumped"] = df["pango_lineage"].where(
    df["pango_lineage"].isin(top_lineages), other="Other"
)
df["seq_prop_zscore"] = (
    (df["wn_prop_sequenced"] - df["wn_prop_sequenced"].mean())
    / df["wn_prop_sequenced"].std()
)

for col in ["median_age", "age_diversity", "simd_quintile_std", "frac_female", "frac_vaccinated"]:
    df[col] = StandardScaler().fit_transform(df[[col]])

In [23]:
waves = [
    'WV1_B.1.177_C108360',
    'WV2_B.1.1.7_C574152',
    'WV3_AY.4_C983568',
     'WV4_BA.2_C479360',
    'WV5_BA.2_C85080',
    'WV6_BA.5.2_C23016'
]

model_data = df[
    (df["resolution"] == data.PRIMARY_RESOLUTION)
    & (df["wave"] == waves[0])
].copy()

model_data["simd_quintile_mode"] = pd.Categorical(
    model_data["simd_quintile_mode"],
    categories=[3, 1, 2, 4, 5],  # reference = 3
    ordered=False,
)

formula = """
n_sequences_minus_one ~
    median_age +
    age_diversity +
    frac_female +
    frac_vaccinated +
    simd_quintile_mode +
    simd_quintile_std +
    seq_prop_zscore
"""

fit = smf.negativebinomial(
    formula=formula,
    data=model_data,
).fit(method="lbfgs", maxiter=200, disp=0)

print(fit.summary())

                       NegativeBinomial Regression Results                       
Dep. Variable:     n_sequences_minus_one   No. Observations:                 8704
Model:                  NegativeBinomial   Df Residuals:                     8693
Method:                              MLE   Df Model:                           10
Date:                   Tue, 28 Apr 2026   Pseudo R-squ.:                  0.3561
Time:                           13:57:08   Log-Likelihood:                -10219.
converged:                          True   LL-Null:                       -15869.
Covariance Type:               nonrobust   LLR p-value:                     0.000
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                  -1.0785      0.061    -17.789      0.000      -1.197      -0.960
simd_quintile_mode[T.1]    -0.2016      0.043     -4.645      0.000 